In [ ]:
!pip install -q tf-keras==2.19.0 h5py==3.11.0

In [ ]:
import tf_keras as keras  # tf-keras içe aktarma – .h5 uzantılı modeller ile uyumlu Keras sürümü
from tf_keras.models import load_model  # tf_keras içerisinde load_model fonksiyonunu içe aktarma, bu modeli açmamızı sağlar
from PIL import Image, ImageOps  # Pillow kütüphanesinden Image ve ImageOps sınıflarını içe aktarıyoruz
import numpy as np
from google.colab import files


In [ ]:
def detect_ai_or_real_cat(image_path):
    # Açıklık sağlamak için bilimsel gösterimi kapatın
    np.set_printoptions(suppress=True)


    model = load_model("keras_model.h5", compile=False)


    class_names = open("labels.txt", "r").readlines()

    # Keras modeline beslenecek diziyi doğru boyutta oluşturun
    data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)

    # Parametre olarak gelen görsel yolunu açın
    image = Image.open(image_path).convert("RGB")

    # Görseli 224x224 boyutuna getirin
    size = (224, 224)
    image = ImageOps.fit(image, size, Image.Resampling.LANCZOS)

    # Görseli NumPy dizisine dönüştürün
    image_array = np.asarray(image)

    # Görsel verisini normalize edin (-1 ile 1 arasında)
    normalized_image_array = (image_array.astype(np.float32) / 127.5) - 1

    # Görseli diziye yerleştirin
    data[0] = normalized_image_array

    # Tahmin yapın
    prediction = model.predict(data)
    index = np.argmax(prediction)

    # Etiket ismini temizleyin (Dosya formatındaki baştaki rakamları ve boşlukları kaldırır)
    class_name = class_names[index].strip()
    if " " in class_name:
        class_name = class_name.split(" ", 1)[1]

    confidence_score = float(prediction[0][index])

    # Sonuçları yazdırın
    print("Sonuç:", class_name)
    print("Güven Skoru:", confidence_score)

    return class_name, confidence_score

In [ ]:
def main():
    path = files.upload()
    path = list(path.keys())[0]

    class_name, score = detect_ai_or_real_cat(path)

    if class_name == 'ai kedi':
        print('Bu bir ai kedi')
    elif class_name == 'kedi':
        print('Bu bir kedi')

In [ ]:
main()